Pull Model: ollama pull qwen3:4b

# Part I: Exploring LLM Capabilities

We test a local LLM (`qwen3:4b` through Ollama) on five task types: question answering, summarization, reasoning, data interpretation, and code generation. For each task we compare a plain prompt with an improved prompt. Then we check how consistent the answers are across repeated runs and try to get the model to hallucinate.

## Setup

In [ ]:
# Claude assisted code

import re
import ollama
import pandas as pd

MODEL = "qwen3:4b"
LAST_THINKING = ""

def ask(prompt, system=None, temperature=0.7, seed=None):
    """Send one prompt to the local model and return its final answer."""
    global LAST_THINKING
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    options = {"temperature": temperature}
    if seed is not None:
        options["seed"] = seed
    resp = ollama.chat(model=MODEL, messages=messages, think=True, options=options)
    LAST_THINKING = resp.message.thinking or ""
    return resp.message.content.strip()

def show(label, prompt, **kw):
    """Print a prompt, the model's answer, and how long its hidden reasoning was."""
    reply = ask(prompt, **kw)
    print(f"=== {label} ===\nPROMPT: {prompt}\n\nREPLY: {reply}\n"
          f"(hidden reasoning: {len(LAST_THINKING.split())} words)\n")
    return reply

## 1. Question answering
One easy factual question and one trick question: no Fields Medal was awarded in 1987 (they were awarded in 1986 and 1990).

In [ ]:
STRICT = "Answer in one sentence. If you are not sure, or the question has a false premise, say so instead of guessing."

for q in ["What is the capital of Australia?", "Who won the Fields Medal in 1987?"]:
    show("plain", q)
    show("strict system prompt", q, system=STRICT)

## 2. Summarization

In [ ]:
TEXT = """Overfitting happens when a machine learning model learns the training data too closely,
including its noise and random fluctuations, instead of the underlying pattern. An overfit model
performs very well on the training set but poorly on new, unseen data. Common signs include a large
gap between training and validation accuracy. Ways to reduce overfitting include collecting more data,
simplifying the model, using regularization such as L1 or L2 penalties, applying dropout in neural
networks, and using cross-validation to choose hyperparameters. Early stopping, where training is
halted once validation performance stops improving, is another widely used technique."""

plain = show("plain", f"Summarize this:\n\n{TEXT}")
constrained = show("constrained", f"Summarize the text below in exactly 3 bullet points, 40 words total at most, for a beginner.\n\n{TEXT}")

print("Word counts -> plain:", len(plain.split()), "| constrained:", len(constrained.split()))

## 3. Reasoning
The bat-and-ball problem: a bat and a ball cost $1.10 together, and the bat costs $1.00 more than the ball. The intuitive answer ($0.10) is wrong; the correct answer is **$0.05**. We compare a direct-answer prompt with a "think step by step" prompt. (qwen3 always reasons internally first; the step-by-step prompt also asks it to show the work in the answer.)

In [ ]:
Q = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"

show("direct", Q + " Reply with only the amount.")
_ = show("step by step", Q + " Think step by step, then give the final answer on the last line.")

## 4. Data interpretation
We give the model a small sales table and ask questions that pandas can answer exactly, then compare the two.

In [ ]:
df = pd.DataFrame({
    "region": ["North", "South", "East", "North", "South", "East"],
    "month":  ["Jan", "Jan", "Jan", "Feb", "Feb", "Feb"],
    "sales":  [1250, 980, 1430, 1610, 1105, 1175],
})
table = df.to_string(index=False)

reply = show("data question",
             f"Here is a sales table:\n\n{table}\n\n"
             "1) Which region has the highest total sales across both months, and what is that total?\n"
             "2) What is the average sales value across all rows?\n"
             "Answer briefly.")

totals = df.groupby("region")["sales"].sum()
print("Ground truth (pandas):")
print(f"  1) {totals.idxmax()} with {totals.max()}  (all totals: {totals.to_dict()})")
print(f"  2) average = {df['sales'].mean():.2f}")

## 5. Code generation
We ask for a function, then run the code against a few tests. The detailed prompt adds the edge cases to the request.

In [ ]:
def run_tests(reply):
    """Pull the first Python code block out of the reply, run it, and test second_largest."""
    m = re.search(r"```(?:python)?\n(.*?)```", reply, re.S)
    src = m.group(1) if m else reply
    env = {}
    try:
        exec(src, env)
        f = env["second_largest"]
        tests = [([3, 1, 4, 1, 5], 4), ([7, 7, 7], None), ([10], None), ([-2, -5, -1], -2)]
        results = [f(list(x)) == want for x, want in tests]
        return f"{sum(results)}/{len(results)} tests passed"
    except Exception as e:
        return f"error: {e!r}"

plain = ask("Write a Python function second_largest(nums) that returns the second largest number in a list.")
detailed = ask("Write a Python function second_largest(nums) that returns the second largest DISTINCT number "
               "in a list of ints. Return None if there is no such number (e.g. [7, 7, 7] or [10]). "
               "Do not modify the input list. Return only one Python code block.")

print("=== plain ===\n", plain, "\n->", run_tests(plain), "\n")
print("=== detailed ===\n", detailed, "\n->", run_tests(detailed))

## 6. Consistency
We ask the same question 5 times with temperature 0.7 (the default, some randomness) and 5 times with temperature 0 and a fixed seed. The correct answer is **4:35 PM**.

In [ ]:
Q = "A train leaves at 2:45 PM and the trip takes 1 hour 50 minutes. What time does it arrive? Reply with only the time."

runs = {
    "temp 0.7": [ask(Q, temperature=0.7) for _ in range(5)],
    "temp 0 (seed 42)": [ask(Q, temperature=0, seed=42) for _ in range(5)],
}
consistency = pd.DataFrame(runs)
display(consistency)
for col in consistency:
    print(f"{col}: {consistency[col].nunique()} distinct answer(s)")

## 7. Hallucination probes
Both questions are about things that do not exist: a made-up paper, and a pandas method `DataFrame.auto_clean()` that is not real. We ask each one with and without a system prompt that allows the model to say it does not know.

In [ ]:
HONEST = "Only state facts you are confident are true. If something does not exist or you don't know, say so plainly."

probes = [
    "Summarize the main findings of the 2018 paper 'Gradient Forests for Temporal Pandas' by Elena Marquez and Raj Patel.",
    "How do I use pandas' DataFrame.auto_clean() method? Give a short example.",
]
for p in probes:
    show("plain", p)
    show("honesty system prompt", p, system=HONEST)

## 8. Summary and discussion

need to do